# 03 Export Viewer Manifest

This notebook converts the analyzed and projected frame table into a compact JSON manifest that the browser app can load later.

Use this when you want the browser to stay lightweight while Python does the heavier computation.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd

INPUT_CSV = Path('analysis_output/projection_clusters.csv')
OUTPUT_JSON = Path('analysis_output/viewer_manifest.json')
OUTPUT_JSON.parent.mkdir(exist_ok=True)

print(f'Input CSV: {INPUT_CSV.resolve()}')
print(f'Output JSON: {OUTPUT_JSON.resolve()}')

In [ ]:
def build_manifest(frame, source_audio=None, source_video=None, projection_method='pca', color_mode='rms', max_points=5000):
    reduced = frame.head(max_points).copy()
    frames = []
    mfcc_columns = [column for column in reduced.columns if column.startswith('mfcc_')]

    for _, row in reduced.iterrows():
        frames.append({
            'time': float(row.get('time', 0.0)),
            'x': float(row.get('x', 0.0)),
            'y': float(row.get('y', 0.0)),
            'z': float(row.get('z', 0.0)),
            'rms': float(row.get('rms', 0.0)),
            'centroid': float(row.get('centroid', 0.0)),
            'flatness': float(row.get('flatness', 0.0)),
            'rolloff': float(row.get('rolloff', 0.0)),
            'bandwidth': float(row.get('bandwidth', 0.0)),
            'zcr': float(row.get('zcr', 0.0)),
            'cluster': int(row.get('cluster', -1)),
            'mfcc': [float(row[column]) for column in mfcc_columns],
        })

    summary = {
        'created_at': datetime.now(timezone.utc).isoformat(),
        'source_audio': str(source_audio) if source_audio else None,
        'source_video': str(source_video) if source_video else None,
        'projection_method': projection_method,
        'color_mode': color_mode,
        'frame_count': len(frame),
        'exported_frame_count': len(frames),
        'columns': list(frame.columns),
    }

    return {
        'version': 1,
        'summary': summary,
        'frames': frames,
    }

In [ ]:
if INPUT_CSV.exists():
    frame = pd.read_csv(INPUT_CSV)
    manifest = build_manifest(frame, projection_method='pca', color_mode='rms')
    with OUTPUT_JSON.open('w', encoding='utf-8') as handle:
        json.dump(manifest, handle, indent=2)
    print(f'Wrote {OUTPUT_JSON}')
    print(f'Frames in export: {manifest[summary][exported_frame_count]}')
else:
    print('Run Notebook 02 first or update INPUT_CSV to point at a projected CSV file.')

## Browser workflow

Once the manifest is exported, you can load it into the browser viewer as a follow-up enhancement.
That lets Python handle the heavy lifting while the web app stays focused on playback and interaction.